### Dataset and Dataloader testing

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

from src.data.dataset import PreferenceDataset, PreferenceCollator, create_dataloader


MODEL_NAME = "gpt2"
MAX_LENGTH = 512
SAMPLE_SIZE = 20


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Load only 20 examples
if SAMPLE_SIZE is not None:
    raw_dataset = load_dataset(
        "Anthropic/hh-rlhf",
        split=f"train[:{SAMPLE_SIZE}]",
    )
else:
    raw_dataset = load_dataset(
        "Anthropic/hh-rlhf",
        split="train",
    )


dataset = PreferenceDataset(
    data=raw_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# ------------------------------------------------------------
# Take a few individual examples
# ------------------------------------------------------------

examples = [
    dataset[0],
    dataset[1],
    dataset[2],
]


# ------------------------------------------------------------
# Create collator
# ------------------------------------------------------------

collator = PreferenceCollator(
    pad_token_id=tokenizer.pad_token_id
)


# ------------------------------------------------------------
# Create a batch manually
# ------------------------------------------------------------

batch = collator(examples)


# ------------------------------------------------------------
# Inspect batch
# ------------------------------------------------------------

print("\nChosen input IDs:")
print(batch["chosen_input_ids"].shape)

print("\nChosen attention mask:")
print(batch["chosen_attention_mask"].shape)

print("\nRejected input IDs:")
print(batch["rejected_input_ids"].shape)

print("\nRejected attention mask:")
print(batch["rejected_attention_mask"].shape)


# ------------------------------------------------------------
# Inspect actual attention masks
# ------------------------------------------------------------

print("\nChosen attention mask:")
print(batch["chosen_attention_mask"])

print("\nRejected attention mask:")
print(batch["rejected_attention_mask"])


Chosen input IDs:
torch.Size([3, 202])

Chosen attention mask:
torch.Size([3, 202])

Rejected input IDs:
torch.Size([3, 196])

Rejected attention mask:
torch.Size([3, 196])

Chosen attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 

In [2]:
dataloader = create_dataloader(
    dataset=dataset,
    tokenizer=tokenizer,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

In [3]:
batch = next(iter(dataloader))

In [4]:
print("\nBatch shapes:")

print(
    "Chosen input IDs:",
    batch["chosen_input_ids"].shape,
)

print(
    "Chosen attention mask:",
    batch["chosen_attention_mask"].shape,
)

print(
    "Rejected input IDs:",
    batch["rejected_input_ids"].shape,
)

print(
    "Rejected attention mask:",
    batch["rejected_attention_mask"].shape,
)


Batch shapes:
Chosen input IDs: torch.Size([4, 202])
Chosen attention mask: torch.Size([4, 202])
Rejected input IDs: torch.Size([4, 196])
Rejected attention mask: torch.Size([4, 196])


In [5]:
print("\nChosen sequence lengths from attention mask:")

print(
    batch["chosen_attention_mask"].sum(dim=1)
)

print("\nRejected sequence lengths from attention mask:")

print(
    batch["rejected_attention_mask"].sum(dim=1)
)


Chosen sequence lengths from attention mask:
tensor([202, 107,  53, 101])

Rejected sequence lengths from attention mask:
tensor([196, 117, 181, 106])


## Reward Model testing

In [6]:
import torch
from transformers import AutoTokenizer

from src.data.dataset import PreferenceDataset
from src.data.dataset import create_dataloader
from src.models.reward_model import GPT2RewardModel

from datasets import load_dataset


MODEL_NAME = "gpt2"
MAX_LENGTH = 512
SAMPLE_SIZE = 20
BATCH_SIZE = 4


# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# Load small dataset
# ============================================================

raw_dataset = load_dataset(
    "Anthropic/hh-rlhf",
    split=f"train[:{SAMPLE_SIZE}]",
)


dataset = PreferenceDataset(
    data=raw_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# ============================================================
# DataLoader
# ============================================================

dataloader = create_dataloader(
    dataset=dataset,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


batch = next(iter(dataloader))


# ============================================================
# Model
# ============================================================

model = GPT2RewardModel(
    model_name=MODEL_NAME
)


# ============================================================
# Forward pass
# ============================================================

with torch.no_grad():

    chosen_rewards = model(
        input_ids=batch["chosen_input_ids"],
        attention_mask=batch["chosen_attention_mask"],
    )

    rejected_rewards = model(
        input_ids=batch["rejected_input_ids"],
        attention_mask=batch["rejected_attention_mask"],
    )


# ============================================================
# Inspect
# ============================================================

print("\nChosen rewards:")
print(chosen_rewards)

print("\nRejected rewards:")
print(rejected_rewards)

print("\nShapes:")

print(
    "chosen_rewards:",
    chosen_rewards.shape,
)

print(
    "rejected_rewards:",
    rejected_rewards.shape,
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

c:\Projects\testenv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manin\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



Chosen rewards:
tensor([-2.3810, -2.2455, -2.7978, -2.1226])

Rejected rewards:
tensor([-2.2680, -1.9963, -2.3768, -2.0792])

Shapes:
chosen_rewards: torch.Size([4])
rejected_rewards: torch.Size([4])


### Bradely Terry Loss testing

In [1]:
import torch

from src.losses.bradley_terry import bradley_terry_loss


# ============================================================
# Case 1: Equal rewards
# ============================================================

chosen = torch.tensor([0.0])
rejected = torch.tensor([0.0])

loss = bradley_terry_loss(
    chosen,
    rejected,
)

print("Case 1 - Equal rewards")
print("Loss:", loss.item())


# ============================================================
# Case 2: Chosen is better
# ============================================================

chosen = torch.tensor([2.0])
rejected = torch.tensor([0.0])

loss = bradley_terry_loss(
    chosen,
    rejected,
)

print("\nCase 2 - Chosen reward higher")
print("Loss:", loss.item())


# ============================================================
# Case 3: Chosen is much better
# ============================================================

chosen = torch.tensor([5.0])
rejected = torch.tensor([0.0])

loss = bradley_terry_loss(
    chosen,
    rejected,
)

print("\nCase 3 - Chosen reward much higher")
print("Loss:", loss.item())


# ============================================================
# Case 4: Rejected is better
# ============================================================

chosen = torch.tensor([0.0])
rejected = torch.tensor([2.0])

loss = bradley_terry_loss(
    chosen,
    rejected,
)

print("\nCase 4 - Rejected reward higher")
print("Loss:", loss.item())

Case 1 - Equal rewards
Loss: 0.6931471824645996

Case 2 - Chosen reward higher
Loss: 0.12692801654338837

Case 3 - Chosen reward much higher
Loss: 0.006715348456054926

Case 4 - Rejected reward higher
Loss: 2.1269280910491943


In [4]:
torch.log(torch.sigmoid(torch.tensor(0)))

tensor(-0.6931)

### test_training_step

In [6]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

from src.data.dataset import PreferenceDataset
from src.data.dataset import create_dataloader
from src.models.reward_model import GPT2RewardModel
from src.losses.bradley_terry import bradley_terry_loss


# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "gpt2"
MAX_LENGTH = 512

SAMPLE_SIZE = 20
BATCH_SIZE = 4

LEARNING_RATE = 1e-5


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# Dataset
# ============================================================

raw_dataset = load_dataset(
    "Anthropic/hh-rlhf",
    split=f"train[:{SAMPLE_SIZE}]",
)

dataset = PreferenceDataset(
    data=raw_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# ============================================================
# DataLoader
# ============================================================

dataloader = create_dataloader(
    dataset=dataset,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


batch = next(iter(dataloader))


# ============================================================
# Move batch to device
# ============================================================

batch = {
    key: value.to(device)
    for key, value in batch.items()
}


# ============================================================
# Model
# ============================================================

model = GPT2RewardModel(
    model_name=MODEL_NAME
)

model = model.to(device)

model.train()


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)


# ============================================================
# Forward pass
# ============================================================

chosen_rewards = model(
    input_ids=batch["chosen_input_ids"],
    attention_mask=batch["chosen_attention_mask"],
)

rejected_rewards = model(
    input_ids=batch["rejected_input_ids"],
    attention_mask=batch["rejected_attention_mask"],
)


print("\nChosen rewards:")
print(chosen_rewards)

print("\nRejected rewards:")
print(rejected_rewards)


# ============================================================
# Bradley-Terry loss
# ============================================================

loss = bradley_terry_loss(
    chosen_rewards=chosen_rewards,
    rejected_rewards=rejected_rewards,
)


print("\nLoss before backward:")
print(loss.item())


# ============================================================
# Backward pass
# ============================================================

optimizer.zero_grad()

loss.backward()


# ============================================================
# Inspect gradients
# ============================================================

reward_head_gradient = (
    model.reward_head.weight.grad
)

print("\nReward head gradient:")
print(reward_head_gradient)

print(
    "\nReward head gradient norm:",
    reward_head_gradient.norm().item(),
)


# ============================================================
# Inspect GPT-2 gradient
# ============================================================

embedding_gradient = (
    model.backbone.wte.weight.grad
)

print(
    "\nGPT-2 embedding gradient norm:",
    embedding_gradient.norm().item(),
)


# ============================================================
# Optimizer step
# ============================================================
old_reward_head = (
    model.reward_head.weight.detach().clone()
)
optimizer.step()

print("\nOptimizer step completed.")
new_reward_head = (
    model.reward_head.weight.detach().clone()
)

parameter_change = (
    new_reward_head - old_reward_head
).norm().item()

print(
    "\nReward head parameter change:",
    parameter_change,
)

Device: cuda

Chosen rewards:
tensor([0.4391, 1.3233, 1.3726, 1.4745], device='cuda:0',
       grad_fn=<SqueezeBackward1>)

Rejected rewards:
tensor([0.6635, 1.9358, 2.3511, 1.3193], device='cuda:0',
       grad_fn=<SqueezeBackward1>)

Loss before backward:
0.9433529376983643

Reward head gradient:
tensor([[-1.0698e-02,  1.4813e-01, -1.7828e-01, -2.6091e-02, -1.2628e-01,
          6.9494e-02, -5.4832e+00, -1.9386e-02, -2.4877e-02,  1.2602e-01,
         -8.8983e-03, -6.4650e-02,  9.2776e-02, -1.1087e-01, -1.1015e-01,
         -1.2342e-03, -1.6594e-01, -3.1516e-01,  6.4919e-02,  1.1380e-01,
          8.4504e-02, -2.4388e-01, -2.6110e-02,  1.8625e-01,  7.6740e-02,
          1.2837e-01,  7.9542e-02, -2.2327e-01,  1.4637e-01, -9.6222e-02,
          2.0065e-01,  1.0977e-01,  1.8048e-01, -4.1110e-02,  1.6498e-01,
         -1.6022e-02, -2.0630e+01,  1.0789e-02,  1.1460e-01, -6.2337e-02,
          1.1169e-01, -3.9597e-02,  1.5554e-02,  2.6289e-01, -3.8633e-02,
          1.8248e-02, -3.8347e-02,

### Trainer testing

In [7]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

from src.data.dataset import PreferenceDataset
from src.data.dataset import create_dataloader
from src.models.reward_model import GPT2RewardModel
from src.training.trainer import RewardModelTrainer


MODEL_NAME = "gpt2"

MAX_LENGTH = 512
SAMPLE_SIZE = 20
BATCH_SIZE = 4

LEARNING_RATE = 1e-5
NUM_EPOCHS = 5


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# --------------------------------------------------
# Tokenizer
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# --------------------------------------------------
# Dataset
# --------------------------------------------------

raw_dataset = load_dataset(
    "Anthropic/hh-rlhf",
    split=f"train[:{SAMPLE_SIZE}]",
)


dataset = PreferenceDataset(
    data=raw_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# --------------------------------------------------
# DataLoader
# --------------------------------------------------

dataloader = create_dataloader(
    dataset=dataset,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)


# --------------------------------------------------
# Model
# --------------------------------------------------

model = GPT2RewardModel(
    model_name=MODEL_NAME
).to(device)


# --------------------------------------------------
# Optimizer
# --------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)


# --------------------------------------------------
# Trainer
# --------------------------------------------------

trainer = RewardModelTrainer(
    model=model,
    optimizer=optimizer,
    device=device,
)


# --------------------------------------------------
# Training
# --------------------------------------------------

for epoch in range(NUM_EPOCHS):

    metrics = trainer.train_epoch(
        dataloader
    )

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} "
        f"| Loss: {metrics['loss']:.4f} "
        f"| Accuracy: {metrics['accuracy']:.4f}"
    )

Device: cuda
Epoch 1/5 | Loss: 1.8168 | Accuracy: 0.2500
Epoch 2/5 | Loss: 0.7053 | Accuracy: 0.7000
Epoch 3/5 | Loss: 0.5020 | Accuracy: 0.8000
Epoch 4/5 | Loss: 0.7001 | Accuracy: 0.7000
Epoch 5/5 | Loss: 0.6658 | Accuracy: 0.8000
